# M05 · Motion profiles - trapezoids, stepper ramps and DDA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/04_control_motion/M05_motion_profiles_trapezoids_stepper_ramps_and_dda.ipynb)

**Track:** Control and motion · **Skill axis (sheet):** Control and motion · **Time:** about 60 min

**Skill Hub concepts:** Trapezoidal motion profile; Stepper acceleration ramp; Digital differential analyzer

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("M05")

## Map: from "go to x" to a stream of setpoints or step pulses

```
 velocity                       trapezoid                      triangle (distance too short to reach v_max)
  v_max  ____/‾‾‾‾‾‾‾‾‾‾\____          v_peak = sqrt(d * a_max)        /\
            a      cruise   -a                                        /  \
```

| Case | Condition | Times |
|---|---|---|
| Trapezoid | $d \ge v_{max}^2 / a_{max}$ | $t_a = v_{max}/a_{max}$, $t_c = (d - v_{max}^2/a_{max})/v_{max}$ |
| Triangle | $d < v_{max}^2 / a_{max}$ | $v_{peak} = \sqrt{d\,a_{max}}$, $t_a = v_{peak}/a_{max}$, $t_c = 0$ |

**Stepper ramp in real time (Austin, 2005):** a timer fires one step every $c_i$ counts. Constant acceleration
$\dot\omega$ with step angle $\alpha$ and timer frequency $f$:
$$c_0 = 0.676\, f \sqrt{2\alpha/\dot\omega}, \qquad c_i = c_{i-1} - \frac{2\,c_{i-1}}{4i + 1}$$
Exact delay for comparison: $c_i^{exact} = f\sqrt{2\alpha/\dot\omega}\,(\sqrt{i+1} - \sqrt{i})$. The recurrence needs one
division per step instead of a square root.

**Digital differential analyzer (Bresenham):** for a coordinated move of $(d_x, d_y)$ steps, the major axis steps
every tick and an error accumulator decides when the minor axis steps, so both finish together on a straight line.

Microstepping: steps per revolution = full steps x microsteps (200 x 16 = 3200).

In [ ]:
import math
import numpy as np

## Task 1: plan a trapezoid and sample it

In [ ]:
def trapezoid(d, v_max, a_max):
    """Return (t_acc, t_const, t_total, v_peak)."""
    raise NotImplementedError  # TODO

def trap_position(t, d, v_max, a_max):
    raise NotImplementedError  # TODO

lab.check("trapezoid", trapezoid, trap_position)

In [ ]:
%%writefile m05_trapezoid.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/m05.h
// Grade it with lab.check("trapezoid", "m05_trapezoid.c"). For C++, name the file .cpp in both places.
#include "m05.h"

void trapezoid(double d, double v_max, double a_max, double *t_acc, double *t_const, double *t_total, double *v_peak) {
    (void)d; (void)v_max; (void)a_max;
    *t_acc = 0; *t_const = 0; *t_total = 0; *v_peak = 0; /* TODO: trapezoid or triangle */
}

double trap_position(double t, double d, double v_max, double a_max) {
    (void)t; (void)d; (void)v_max; (void)a_max;
    return 0; /* TODO */
}

In [ ]:
# lab.check("trapezoid", "m05_trapezoid.c")  # uncomment after writing the C/C++ cell above

## Task 2: the stepper acceleration ramp

In [ ]:
def austin_delays(n, f_timer, alpha, accel):
    raise NotImplementedError  # TODO

lab.check("ramp", austin_delays)

In [ ]:
%%writefile m05_ramp.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/m05.h
// Grade it with lab.check("ramp", "m05_ramp.c"). For C++, name the file .cpp in both places.
#include "m05.h"

void austin_delays(size_t n, double f_timer, double alpha, double accel, double *counts) {
    (void)f_timer; (void)alpha; (void)accel;
    for (size_t i = 0; i < n; ++i) counts[i] = 0; /* TODO: c0 = 0.676 f sqrt(2 alpha / a), then the recurrence */
}

In [ ]:
# lab.check("ramp", "m05_ramp.c")  # uncomment after writing the C/C++ cell above

## Task 3: coordinated two-axis stepping

In [ ]:
def dda(dx, dy):
    """dx, dy >= 0. One (step_x, step_y) pair per tick of the major axis."""
    raise NotImplementedError  # TODO

lab.check("dda", dda)

In [ ]:
%%writefile m05_dda.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/m05.h
// Grade it with lab.check("dda", "m05_dda.c"). For C++, name the file .cpp in both places.
#include "m05.h"

size_t dda(uint32_t dx, uint32_t dy, uint8_t *step_x, uint8_t *step_y, size_t cap) {
    (void)dx; (void)dy; (void)step_x; (void)step_y; (void)cap;
    return 0; /* TODO: one step on the major axis per tick; accumulate the minor axis error */
}

In [ ]:
# lab.check("dda", "m05_dda.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** $d = 1.0$ m, $v_{max} = 0.5$ m/s, $a_{max} = 1.0$ m/s$^2$. Total move time in seconds?

**M2 (Apply).** $d = 0.1$ m, $v_{max} = 1$ m/s, $a_{max} = 1$ m/s$^2$. Peak velocity?

**M3 (Apply).** 200-step motor at 1/16 microstepping turning 1 rev/s. Step pulses per second?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** Why ramp a stepper instead of starting at full speed?
- a) to save power
- b) torque available is limited; demanding infinite acceleration makes the rotor fall behind the field and miss steps
- c) drivers require it

**K2 (Evaluate).** When would you move from a trapezoid to an S-curve (jerk-limited) profile, and what does a DDA buy
you over running one independent timer per axis?

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `m05.h`).
3. In `roboprep/checks/m05.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()